# 03 - Modelling and Comparison

**Coursework Part 01, Task 2.II**

**Goal:** train, tune and compare three classifiers — **K-Nearest Neighbours (KNN)**, **Logistic Regression** and **Random Forest** — for predicting customer churn, and identify the most suitable model.

**Input:** `data/train.csv` and `data/test.csv` from `02_preprocessing.ipynb` (cleaned, unencoded, `Churn` = 1/0).

This notebook defines the experiments (models, hyperparameter grids, settings) and interprets the results.

## Setup

In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_validate, cross_val_predict
from sklearn.metrics import (classification_report, precision_recall_curve,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

from src.cm2604_ml_cw.preprocessing import encode
from src.cm2604_ml_cw.evaluation import *

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

TRAIN_PATH = Path("../data/processed/train.csv")
TEST_PATH = Path("../data/processed/test.csv")

RANDOM_STATE = 20080337
SAVE_FIGS = True            # set to False if you don't want PNGs in figures/

## Load The Data

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

X_train, y_train = train.drop(columns="Churn"), train["Churn"]
X_test, y_test = test.drop(columns="Churn"), test["Churn"]

pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "features": [X_train.shape[1], X_test.shape[1]],
    "churn_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]).style.format({"churn_rate": "{:.2%}"})

,rows,features,churn_rate
train,5634,19,26.54%
test,1409,19,26.54%


## Encoding And Scaling

In [3]:
# KNN and Logistic Regression: encoding + scaling
X_train_scaled, X_test_scaled, preprocessor_scaled = encode(X_train, X_test, scale=True)

Scaling applied         : True
Features before encoding: 19
Features after encoding : 40

['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'SeniorCitizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Cre

In [4]:
# Random Forest: encoding only
X_train_rf, X_test_rf, preprocessor_rf = encode(X_train, X_test, scale=False, verbose=False)
print("Random Forest features:", X_train_rf.shape[1])
X_train_rf[["tenure", "MonthlyCharges", "TotalCharges"]].head(3)

Random Forest features: 40


,tenure,MonthlyCharges,TotalCharges
0,20,20.35,393.15
1,72,97.25,7133.10
2,23,84.40,1936.85


## Evaluation Strategy

- **Primary metric: F1-score for the churn class.** The data is imbalanced (26.5% churn), so accuracy is misleading: predicting "no churn" for everyone already gives ~73%. F1 balances **precision** and **recall**.
- **Also reported:** recall, precision, ROC-AUC and PR-AUC.
- **Tuning:** stratified 5-fold cross-validation on the **training set only**, so every fold keeps the 26.5% churn rate. The test set is used once, at the end.
- **Class imbalance:** handled with `class_weight="balanced"` for Logistic Regression and Random Forest. Mistakes on churners are weighted more heavily, in proportion to how rare they are. KNN has no class-weight option, so its imbalance is handled through the choice of metric.

The helper functions (`results_table`, `best_scores`, `plot_validation_curve`, `evaluate`, `save_fig`) are in `src/cm2604_ml_cw/evaluation.py`.

In [5]:
# Stratified 5-fold CV used for every model
cv = make_cv(RANDOM_STATE)

# Metrics recorded during cross-validation (defined in src/evaluation.py)
list(SCORING)

['f1', 'recall', 'precision', 'roc_auc']

## Baseline And Default Models

Before tuning, each model is evaluated with its default hyperparameters, alongside a **majority-class baseline** that always predicts "no churn". This gives a reference point for measuring the effect of tuning.

In [6]:
default_models = {
    "Baseline (majority class)": (DummyClassifier(strategy="most_frequent"), X_train_scaled),
    "KNN": (KNeighborsClassifier(), X_train_scaled),
    "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), X_train_scaled),
    "Random Forest": (RandomForestClassifier(random_state=RANDOM_STATE), X_train_rf),
}

rows = {}
for name, (model, X) in default_models.items():
    scores = cross_validate(model, X, y_train, cv=cv, n_jobs=-1,
                            scoring={"accuracy": "accuracy", **SCORING})
    rows[name] = {
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc": scores["test_roc_auc"].mean(),
    }

default_cv = pd.DataFrame(rows).T.round(4)
default_cv

,accuracy,precision,recall,f1,f1_std,roc_auc
Baseline (majority class),0.7346,0.0000,0.0000,0.0000,0.0000,0.5000
KNN,0.7675,0.5673,0.5264,0.5457,0.0126,0.7790
Logistic Regression,0.8040,0.6564,0.5505,0.5986,0.0138,0.8422
Random Forest,0.7891,0.6353,0.4816,0.5476,0.0209,0.8180


**Observations**
- The baseline reaches **73.5% accuracy with 0 recall and 0 F1**: it never identifies a single churner. This confirms that accuracy is not a suitable metric for this problem.
- All three default models beat the baseline, but **none handles the imbalance**. Their recall is only 0.50-0.55, so they miss about half of the churners. With 73% of customers staying, the default 0.5 threshold pushes them towards predicting "no churn".
- **Logistic Regression** is the strongest default model (CV F1 0.595, ROC-AUC 0.844).
- The default **Random Forest** has high accuracy (0.791) but the lowest recall (0.500).
- The default **KNN** (k = 5) has the lowest ROC-AUC (0.780). A small k makes it sensitive to noise in individual customers.